# 🥇 Camada Gold — Analytical

A camada Gold entrega dados **agregados e prontos para consumo analítico**. Aqui calculamos os principais KPIs de produção industrial, incluindo o **OEE (Overall Equipment Effectiveness)** — o indicador mais importante em ambientes de manufatura.

**Agregações por:** `line_id` + `shift` + `equipment_id`

**Métricas calculadas:**

| Métrica | Fórmula | Descrição |
|---|---|---|
| `availability_pct` | `(turno_min - downtime) / turno_min × 100` | % do tempo em que o equipamento esteve disponível |
| `quality_pct` | `(produção - defeitos) / produção × 100` | % de produtos dentro da especificação |
| `oee_pct` | `availability × quality × 100` | Eficiência global do equipamento |
| `defect_rate_pct` | `defeitos / produção × 100` | Taxa de não conformidade |

> 📌 OEE acima de 85% é considerado classe mundial em manufatura. Abaixo de 60% indica necessidade de ação imediata.

## 1. Criando o schema Gold

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

## 2. Criação da tabela de KPIs

Agregação dos dados da Silver com cálculo de OEE e métricas de produção.

> `480 minutos` = duração padrão de um turno de 8 horas

In [0]:
%sql
CREATE OR REPLACE TABLE gold.production_kpis AS
SELECT
  line_id,
  shift,
  equipment_id,

  -- Volume de produção
  SUM(production_qty)             AS total_production,
  SUM(defect_qty)                 AS total_defects,
  SUM(downtime_minutes)           AS total_downtime_min,
  ROUND(AVG(downtime_minutes), 2) AS avg_downtime_min,
  COUNT(*)                        AS event_count,

  -- Taxa de defeitos: defeitos / produção total
  ROUND(SUM(defect_qty) / NULLIF(SUM(production_qty), 0) * 100, 2) AS defect_rate_pct,

  -- Availability: (tempo do turno - downtime) / tempo do turno
  ROUND(
    (COUNT(*) * 480 - SUM(downtime_minutes)) / (COUNT(*) * 480) * 100
  , 2) AS availability_pct,

  -- Quality: (produção - defeitos) / produção
  ROUND(
    (SUM(production_qty) - SUM(defect_qty)) / NULLIF(SUM(production_qty), 0) * 100
  , 2) AS quality_pct,

  -- OEE = Availability × Quality (Performance assumida como 100%)
  ROUND(
    ((COUNT(*) * 480 - SUM(downtime_minutes)) / (COUNT(*) * 480)) *
    ((SUM(production_qty) - SUM(defect_qty)) / NULLIF(SUM(production_qty), 0)) * 100
  , 2) AS oee_pct

FROM silver.production_events
GROUP BY line_id, shift, equipment_id
ORDER BY line_id, shift, equipment_id;

## 3. Visualização dos KPIs gerados

In [0]:
%sql
SELECT * FROM gold.production_kpis;